In [22]:
# =========================================================
# ELASTIC NET REGRESSION FROM SCRATCH
# MULTIPLE LINEAR REGRESSION
# USING BATCH GRADIENT DESCENT
# =========================================================

import numpy as np

from sklearn.datasets import load_diabetes


class ElasticNet_MLR:

    # =====================================================
    # CONSTRUCTOR
    # =====================================================
    def __init__(
        self,
        learning_rate=0.001,
        epochs=1000,
        alpha=0.1,
        l1_ratio=0.5
    ):

        self.learning_rate = learning_rate
        self.epochs = epochs

        # Overall Regularization Strength
        self.alpha = alpha

        # L1 vs L2 Mixing
        self.l1_ratio = l1_ratio

        self.intercept_ = 0
        self.coef_ = None


    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    def train_test_split(
        self,
        X,
        y,
        test_size=0.2,
        random_state=None
    ):

        X = np.array(X)
        y = np.array(y)

        n = len(X)

        indices = np.arange(n)

        np.random.seed(random_state)
        np.random.shuffle(indices)

        X = X[indices]
        y = y[indices]

        split_index = int(n * (1 - test_size))

        X_train = X[:split_index]
        X_test = X[split_index:]

        y_train = y[:split_index]
        y_test = y[split_index:]

        return X_train, X_test, y_train, y_test


    # =====================================================
    # FIT MODEL USING ELASTIC NET OBJECTIVE
    # =====================================================
    def fit(self, X, y):

        X = np.array(X)
        y = np.array(y)

        n_samples = X.shape[0]
        n_features = X.shape[1]

        # Initialize Parameters
        self.coef_ = np.zeros(n_features)

        self.intercept_ = 0


        # =================================================
        # TRAINING LOOP
        # =================================================
        for epoch in range(self.epochs):

            # Predictions
            y_pred = (
                np.dot(X, self.coef_)
                + self.intercept_
            )

            # =================================================
            # OLS GRADIENT
            # =================================================
            d_coef = (
                (-2 / n_samples)
                * np.dot(X.T, (y - y_pred))
            )

            # =================================================
            # L1 GRADIENT
            # =================================================
            l1_gradient = (
                self.alpha
                * self.l1_ratio
                * np.sign(self.coef_)
            )

            # =================================================
            # L2 GRADIENT
            # =================================================
            l2_gradient = (
                2
                * self.alpha
                * (1 - self.l1_ratio)
                * self.coef_
            )

            # =================================================
            # TOTAL GRADIENT
            # =================================================
            d_coef = (
                d_coef
                + l1_gradient
                + l2_gradient
            )

            # Intercept Gradient
            d_intercept = (
                (-2 / n_samples)
                * np.sum(y - y_pred)
            )

            # =================================================
            # UPDATE PARAMETERS
            # =================================================
            self.coef_ = (
                self.coef_
                - self.learning_rate * d_coef
            )

            self.intercept_ = (
                self.intercept_
                - self.learning_rate * d_intercept
            )


    # =====================================================
    # PREDICTION
    # =====================================================
    def predict(self, X):

        X = np.array(X)

        y_pred = (
            np.dot(X, self.coef_)
            + self.intercept_
        )

        return y_pred


    # =====================================================
    # EVALUATION METRICS
    # =====================================================
    def evaluation(self, y_true, y_pred, X_test):

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        # MSE
        mse = np.mean((y_true - y_pred) ** 2)

        # RMSE
        rmse = np.sqrt(mse)

        # MAE
        mae = np.mean(np.abs(y_true - y_pred))

        # R2 Score
        ss_total = np.sum(
            (y_true - np.mean(y_true)) ** 2
        )

        ss_residual = np.sum(
            (y_true - y_pred) ** 2
        )

        r2 = 1 - (ss_residual / ss_total)

        # Adjusted R2
        n = len(y_true)
        p = X_test.shape[1]

        adjusted_r2 = 1 - (
            ((1 - r2) * (n - 1))
            / (n - p - 1)
        )

        # Print Metrics
        print("MSE         :", mse)

        print("RMSE        :", rmse)

        print("MAE         :", mae)

        print("R2 Score    :", r2)

        print("Adjusted R2 :", adjusted_r2)


# =========================================================
# LOAD DIABETES DATASET
# =========================================================

diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target


# =========================================================
# MODEL
# =========================================================

model = ElasticNet_MLR(
    learning_rate=0.001,
    epochs=1000,
    alpha=0.1,
    l1_ratio=0.5
)


# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = model.train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# =========================================================
# TRAIN MODEL
# =========================================================

model.fit(X_train, y_train)


# =========================================================
# PREDICTION
# =========================================================

y_pred = model.predict(X_test)


# =========================================================
# PARAMETERS
# =========================================================

print("Intercept :")
print(model.intercept_)

print()

print("Coefficients :")
print(model.coef_)


# =========================================================
# EVALUATION
# =========================================================

model.evaluation(y_test, y_pred, X_test)


# =========================================================
# IMPORTANT CONCEPTS
# =========================================================

# Model:
# Multiple Linear Regression

# Objective:
#
# min ||y - Xb||²
# + λ1||b||₁
# + λ2||b||²

# Regularization:
# L1 + L2 Regularization

# Optimization:
# Batch Gradient Descent


# WHY ELASTIC NET?
# ---------------------------------------------------------

# ElasticNet combines:
#
# 1. Ridge Regression (L2)
# 2. Lasso Regression (L1)


# EFFECT OF ELASTIC NET
# ---------------------------------------------------------

# ElasticNet:
#
# - Shrinks coefficients
# - Can perform feature selection
# - Handles multicollinearity


# IMPORTANT PARAMETERS
# ---------------------------------------------------------

# alpha
#
# Overall regularization strength.


# l1_ratio
# ---------------------------------------------------------

# l1_ratio = 1
# -> Pure Lasso
#
# l1_ratio = 0
# -> Pure Ridge
#
# l1_ratio = 0.5
# -> Equal mix of L1 and L2


# ELASTIC NET GRADIENT
# ---------------------------------------------------------

# Total Gradient:
#
# OLS Gradient
# + L1 Gradient
# + L2 Gradient


# IMPORTANT NOTE
# ---------------------------------------------------------

# sklearn internally mainly uses:
#
# Coordinate Descent
#
# because ElasticNet optimization
# involves L1 regularization.


# INTERCEPT
# ---------------------------------------------------------

# Intercept is usually NOT regularized.

Intercept :
134.107279461026

Coefficients :
[ 1.593019    0.42672387  4.17047262  2.77468656  1.27089798  1.11468314
 -3.27898054  3.40563384  4.17619475  2.73098939]
MSE         : 5548.367639311583
RMSE        : 74.48736563546588
MAE         : 61.3074734461612
R2 Score    : 0.0034821383929677374
Adjusted R2 : -0.12427656181306213
